# 02 多项式插值

本节讨论第二章的主要全局多项式工具：

* 拉格朗日插值（Lagrange interpolation）的构造来源；
* 牛顿差商（Newton divided differences）的递推定义、差商表和增量形式；
* Hermite 插值的基本思想：同时使用函数值和导数值；
* 等距节点上的 Runge 现象；
* 切比雪夫节点带来的改进。

拉格朗日插值的核心思想是构造一组“开关函数”：第 $j$ 个基函数只在第 $j$ 个节点处取 1，在其他节点处取 0。牛顿插值和 Hermite 插值可以看成在这个基础上的两种重要组织方式：前者强调递推和新增节点，后者强调把导数信息也纳入插值条件。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    chebyshev_nodes,
    divided_difference_table,
    lagrange_interpolate,
    newton_divided_differences,
    newton_interpolate,
)


## 从两个点开始

已知两个点

$$
(x_0,f_0),\qquad (x_1,f_1),
$$

通过这两个点的直线可以写成

$$
p_1(x)=f_0\frac{x-x_1}{x_0-x_1}
+f_1\frac{x-x_0}{x_1-x_0}.
$$

这个写法的关键在于

$$
\frac{x-x_1}{x_0-x_1}
$$

在 $x=x_0$ 处等于 1，在 $x=x_1$ 处等于 0；而

$$
\frac{x-x_0}{x_1-x_0}
$$

在 $x=x_0$ 处等于 0，在 $x=x_1$ 处等于 1。因此这两个因子就像两个开关，分别控制 $f_0$ 和 $f_1$ 是否被选中。


In [ ]:
x0, x1 = 0.0, 2.0
f0, f1 = 1.0, 3.0
x_line = np.linspace(-0.2, 2.2, 300)

ell0 = (x_line - x1) / (x0 - x1)
ell1 = (x_line - x0) / (x1 - x0)
p1 = f0 * ell0 + f1 * ell1

plt.figure(figsize=(8, 4.5))
plt.plot(x_line, ell0, label=r"$\ell_0(x)$")
plt.plot(x_line, ell1, label=r"$\ell_1(x)$")
plt.plot(x_line, p1, label=r"$p_1(x)$")
plt.scatter([x0, x1], [f0, f1], color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("两点插值中的开关函数")
plt.legend()
plt.tight_layout()
plt.show()


## 一般情形：构造第 j 个基函数

现在给定 $N+1$ 个互异节点

$$
x_0,x_1,\dots,x_N.
$$

希望构造第 $j$ 个基函数 $\ell_j(x)$，使得

$$
\ell_j(x_i)=
\begin{cases}
1, & i=j,\\
0, & i\neq j.
\end{cases}
$$

为了让它在所有其他节点 $x_i,\ i\neq j$ 处为零，最自然的做法是让它含有因子

$$
\prod_{\substack{m=0\\m\neq j}}^N (x-x_m).
$$

这样只要 $x=x_i$ 且 $i\neq j$，其中就有一个因子变成 0。接下来再让它在 $x_j$ 处等于 1，需要除以

$$
\prod_{\substack{m=0\\m\neq j}}^N (x_j-x_m).
$$

因此得到拉格朗日基函数

$$
\boxed{
\ell_j(x)=
\prod_{\substack{m=0\\m\neq j}}^N
\frac{x-x_m}{x_j-x_m}
}.
$$


In [ ]:
def lagrange_basis(nodes, basis_index, x_eval):
    values = np.ones_like(x_eval, dtype=float)
    xj = nodes[basis_index]
    for m, xm in enumerate(nodes):
        if m != basis_index:
            values *= (x_eval - xm) / (xj - xm)
    return values

nodes = np.array([-1.0, -0.3, 0.4, 1.0])
x_plot = np.linspace(-1.1, 1.1, 500)

plt.figure(figsize=(8, 4.5))
for j in range(nodes.size):
    plt.plot(x_plot, lagrange_basis(nodes, j, x_plot), label=rf"$\ell_{j}(x)$")
plt.scatter(nodes, np.zeros_like(nodes), color="black", zorder=3)
plt.xlabel("x")
plt.ylabel("基函数值")
plt.title("拉格朗日基函数：每个基函数负责一个节点")
plt.legend()
plt.tight_layout()
plt.show()


## 拼出插值多项式

若 $\ell_j(x_i)=\delta_{ij}$，则

$$
p_N(x)=\sum_{j=0}^{N} f_j\ell_j(x)
$$

在节点 $x_i$ 上满足

$$
p_N(x_i)=
\sum_{j=0}^{N} f_j\ell_j(x_i)
=
\sum_{j=0}^{N} f_j\delta_{ij}
=f_i.
$$

所以拉格朗日插值多项式为

$$
\boxed{
p_N(x)=
\sum_{j=0}^{N}
f_j
\prod_{\substack{m=0\\m\neq j}}^{N}
\frac{x-x_m}{x_j-x_m}
}.
$$


## 三点例子

给定三个点

$$
(x_0,f_0),\qquad (x_1,f_1),\qquad (x_2,f_2),
$$

二次插值多项式为

$$
p_2(x)=f_0\ell_0(x)+f_1\ell_1(x)+f_2\ell_2(x),
$$

其中

$$
\ell_0(x)=
\frac{(x-x_1)(x-x_2)}
{(x_0-x_1)(x_0-x_2)},
$$

$$
\ell_1(x)=
\frac{(x-x_0)(x-x_2)}
{(x_1-x_0)(x_1-x_2)},
$$

$$
\ell_2(x)=
\frac{(x-x_0)(x-x_1)}
{(x_2-x_0)(x_2-x_1)}.
$$

这些基函数在节点上的取值正好组成单位矩阵，因此插值多项式严格通过三个给定点。


In [ ]:
nodes_three = np.array([-1.0, 0.2, 1.0])
values_three = np.array([1.0, -0.4, 0.6])
x_three = np.linspace(-1.1, 1.1, 400)

basis_at_nodes = np.array([
    [lagrange_basis(nodes_three, j, xi) for j in range(nodes_three.size)]
    for xi in nodes_three
])
print("三点例子中，基函数在节点上的取值矩阵：")
print(np.round(basis_at_nodes, 6))

p2 = lagrange_interpolate(nodes_three, values_three, x_three)

plt.figure(figsize=(8, 4.5))
for j in range(nodes_three.size):
    plt.plot(x_three, lagrange_basis(nodes_three, j, x_three), "--", alpha=0.75, label=rf"$\ell_{j}(x)$")
plt.plot(x_three, p2, color="black", linewidth=2, label=r"$p_2(x)$")
plt.scatter(nodes_three, values_three, color="red", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("三点拉格朗日插值示例")
plt.legend()
plt.tight_layout()
plt.show()


## 为什么插值多项式唯一？

假设 $p_N(x)$ 和 $q_N(x)$ 都是不超过 $N$ 次的多项式，并且都通过这 $N+1$ 个节点。令

$$
r(x)=p_N(x)-q_N(x).
$$

那么 $r(x)$ 仍然是不超过 $N$ 次的多项式，而且

$$
r(x_j)=0,\qquad j=0,1,\dots,N.
$$

也就是说，$r(x)$ 有 $N+1$ 个互异零点。一个不超过 $N$ 次的非零多项式最多只有 $N$ 个零点，所以只能有 $r(x)\equiv 0$。因此 $p_N(x)=q_N(x)$，插值多项式唯一。


## 与切比雪夫微分矩阵的关系

在切比雪夫配置法中，常用节点为

$$
x_j=\cos\frac{j\pi}{N},\qquad j=0,1,\dots,N.
$$

先用这些节点构造拉格朗日插值

$$
p_N(x)=\sum_{j=0}^N f(x_j)\ell_j(x),
$$

再对插值多项式求导：

$$
p_N'(x_i)=\sum_{j=0}^N f(x_j)\ell_j'(x_i).
$$

于是可以定义

$$
D_{ij}=\ell_j'(x_i),
$$

这就是切比雪夫微分矩阵的基本来源。逻辑链条是

$$
\boxed{
\text{节点函数值}
\rightarrow
\text{拉格朗日插值多项式}
\rightarrow
\text{对插值多项式求导}
\rightarrow
\text{微分矩阵}
}.
$$

本章只说明这个来源，具体的矩阵公式、稳定性和谱精度会留到后续“谱方法”章节。


## 牛顿差商：从递推表到插值多项式

拉格朗日形式适合说明“为什么插值多项式存在”，但如果不断加入新节点，拉格朗日形式需要重新组织所有基函数。牛顿插值（Newton interpolation）的优势是把多项式写成逐层增加的形式：

$$
p_N(x)=c_0+c_1(x-x_0)+c_2(x-x_0)(x-x_1)+\cdots+c_N\prod_{m=0}^{N-1}(x-x_m).
$$

其中系数由差商递推得到。先定义零阶差商

$$
f[x_i]=f_i,
$$

再定义高阶差商

$$
f[x_i,x_{i+1},\dots,x_{i+k}]
=
\frac{
f[x_{i+1},\dots,x_{i+k}]-f[x_i,\dots,x_{i+k-1}]
}{x_{i+k}-x_i}.
$$

于是牛顿系数就是

$$
c_k=f[x_0,x_1,\dots,x_k].
$$

这个递推表有两个重要含义：

1. 只要新增一个节点，原来的低阶系数仍然保留，只需要追加新的高阶差商；
2. 求值时可以用类似 Horner 法的嵌套乘法，从最高阶系数向低阶系数回代，避免显式展开多项式。


In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 0.0, 4.0])
x_eval = np.linspace(0.0, 3.0, 300)

lagrange_values = lagrange_interpolate(x, y, x_eval)
newton_nodes, coefficients = newton_divided_differences(x, y)
newton_values = newton_interpolate(newton_nodes, coefficients, x_eval)
_, table = divided_difference_table(x, y)

print("牛顿插值系数 c_k:", coefficients)
print("差商表（每一列是一阶更高的差商）:")
for i in range(x.size):
    entries = []
    for order in range(x.size - i):
        entries.append(f"{table[i, order]: .6f}")
    print(f"x_{i}={x[i]:.1f}: " + "  ".join(entries))

base_nodes, base_coefficients = newton_divided_differences(x[:3], y[:3])
print("前三个节点得到的系数:", base_coefficients)
print("加入第四个节点后，前三个系数仍为:", coefficients[:3])
print("新节点只追加一个新的最高阶系数:", coefficients[3])

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, lagrange_values, label="拉格朗日形式")
plt.plot(x_eval, newton_values, "--", label="牛顿形式")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("p(x)")
plt.title("牛顿形式与拉格朗日形式表示同一个插值多项式")
plt.legend()
plt.tight_layout()
plt.show()


## Hermite 插值：把导数也作为插值条件

前面的插值只使用函数值。如果还知道节点处的导数，就可以构造 Hermite 插值。最基本的条件是

$$
p(x_i)=f_i,\qquad p'(x_i)=m_i.
$$

如果有 $N+1$ 个节点，并且每个节点同时给出函数值和一阶导数，那么一共有 $2N+2$ 个条件。全局 Hermite 多项式的次数通常不超过 $2N+1$。

在数值计算中，更常用的是单区间三次 Hermite 插值。令

$$
t=\frac{x-x_0}{x_1-x_0},\qquad h=x_1-x_0,
$$

则

$$
p(x)=h_{00}(t)f_0+h_{10}(t)h m_0+h_{01}(t)f_1+h_{11}(t)h m_1,
$$

其中

$$
h_{00}(t)=2t^3-3t^2+1,
$$

$$
h_{10}(t)=t^3-2t^2+t,
$$

$$
h_{01}(t)=-2t^3+3t^2,
$$

$$
h_{11}(t)=t^3-t^2.
$$

这说明 Hermite 插值不只是“穿过点”，还要匹配节点处的切线方向。后续的 PCHIP 和分段三次 Hermite 方法，就是围绕“怎样选择节点导数”展开的。


In [ ]:
def cubic_hermite_interval(x0, x1, f0, f1, m0, m1, x_eval):
    h = x1 - x0
    t = (x_eval - x0) / h
    h00 = 2 * t**3 - 3 * t**2 + 1
    h10 = t**3 - 2 * t**2 + t
    h01 = -2 * t**3 + 3 * t**2
    h11 = t**3 - t**2
    return h00 * f0 + h10 * h * m0 + h01 * f1 + h11 * h * m1

x0, x1 = 0.0, np.pi / 2
f0, f1 = np.sin(x0), np.sin(x1)
m0, m1 = np.cos(x0), np.cos(x1)
x_hermite = np.linspace(x0, x1, 300)
y_hermite = cubic_hermite_interval(x0, x1, f0, f1, m0, m1, x_hermite)

plt.figure(figsize=(8, 4.5))
plt.plot(x_hermite, np.sin(x_hermite), label="真实函数 sin(x)")
plt.plot(x_hermite, y_hermite, "--", label="三次 Hermite 插值")
plt.scatter([x0, x1], [f0, f1], color="black", zorder=3, label="函数值")
plt.quiver(
    [x0, x1],
    [f0, f1],
    [0.25, 0.25],
    [0.25 * m0, 0.25 * m1],
    angles="xy",
    scale_units="xy",
    scale=1,
    color="tab:red",
    width=0.004,
    label="端点导数方向",
)
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("单区间三次 Hermite 插值同时匹配函数值和导数")
plt.legend()
plt.tight_layout()
plt.show()


## Runge 现象：为什么等距节点会失败？

Runge 现象不是“多项式插值一定不好”，而是说明：**高次全局多项式 + 等距节点** 这个组合可能非常不稳定。经典测试函数是

$$
f(x)=\frac{1}{1+25x^2},\qquad x\in[-1,1].
$$

对足够光滑的函数，插值误差有形式

$$
f(x)-p_N(x)=\frac{f^{(N+1)}(\xi)}{(N+1)!}\omega_{N+1}(x),
\qquad
\omega_{N+1}(x)=\prod_{j=0}^{N}(x-x_j).
$$

这个公式说明误差至少受两部分控制：

1. 函数自身的高阶导数；
2. 节点乘积 $\omega_{N+1}(x)$ 的大小。

等距节点的问题在于：当 $N$ 增大时，节点乘积和插值基函数在端点附近会变得很大。即使插值多项式在所有节点处都精确等于函数值，节点之间尤其是靠近端点的区域仍可能出现剧烈振荡。

另一个更稳定的视角是看 Lebesgue 函数

$$
\Lambda_N(x)=\sum_{j=0}^{N}|\ell_j(x)|.
$$

它衡量插值算子会怎样放大数据误差或舍入误差。如果 $\max_x \Lambda_N(x)$ 很大，那么节点值中很小的扰动也可能在插值曲线上被明显放大。等距节点的 Lebesgue 常数增长很快，而切比雪夫节点的增长慢得多。


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_plot = np.linspace(-1.0, 1.0, 1200)
y_true = runge(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, n in zip(axes, [9, 17]):
    x_equal = np.linspace(-1.0, 1.0, n)
    y_equal = runge(x_equal)
    y_interp = lagrange_interpolate(x_equal, y_equal, x_plot)
    max_error = np.max(np.abs(y_interp - y_true))

    ax.plot(x_plot, y_true, color="black", label="Runge 函数")
    ax.plot(x_plot, y_interp, label=f"等距节点，节点数={n}")
    ax.scatter(x_equal, y_equal, s=18, zorder=3)
    ax.set_xlabel("x")
    ax.set_title(f"{n - 1} 次插值，多项式最大误差 {max_error:.2e}")
axes[0].set_ylabel("函数值")
axes[0].legend()
fig.suptitle("Runge 现象：等距节点上的端点振荡")
fig.tight_layout()
plt.show()


## 节点乘积和 Lebesgue 函数的对比

下面的实验不只画插值曲线，而是直接比较两个量：

* 节点乘积 $|\omega_{N+1}(x)|$；
* Lebesgue 函数 $\Lambda_N(x)=\sum_j |\ell_j(x)|$。

这两个量越大，插值越容易在节点之间放大误差。注意它们通常在端点附近变大，这正对应 Runge 现象中最明显的振荡区域。


In [ ]:
def nodal_product(nodes, x_eval):
    product = np.ones_like(x_eval, dtype=float)
    for node in nodes:
        product *= x_eval - node
    return product


def lebesgue_function(nodes, x_eval):
    total = np.zeros_like(x_eval, dtype=float)
    for j in range(nodes.size):
        total += np.abs(lagrange_basis(nodes, j, x_eval))
    return total

n = 17
x_equal = np.linspace(-1.0, 1.0, n)
x_cheb = chebyshev_nodes(-1.0, 1.0, n)

omega_equal = np.abs(nodal_product(x_equal, x_plot))
omega_cheb = np.abs(nodal_product(x_cheb, x_plot))
lambda_equal = lebesgue_function(x_equal, x_plot)
lambda_cheb = lebesgue_function(x_cheb, x_plot)

print(f"等距节点最大 Lebesgue 函数值: {np.max(lambda_equal):.3e}")
print(f"切比雪夫节点最大 Lebesgue 函数值: {np.max(lambda_cheb):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(x_plot, omega_equal, label="等距节点")
axes[0].semilogy(x_plot, omega_cheb, label="切比雪夫节点")
axes[0].set_xlabel("x")
axes[0].set_ylabel(r"$|\omega_{N+1}(x)|$")
axes[0].set_title("节点乘积的大小")
axes[0].legend()

axes[1].semilogy(x_plot, lambda_equal, label="等距节点")
axes[1].semilogy(x_plot, lambda_cheb, label="切比雪夫节点")
axes[1].set_xlabel("x")
axes[1].set_ylabel(r"$\Lambda_N(x)$")
axes[1].set_title("Lebesgue 函数的大小")
axes[1].legend()

fig.suptitle("Runge 现象背后的误差放大机制")
fig.tight_layout()
plt.show()


## 切比雪夫节点为什么能缓解？

切比雪夫节点（Chebyshev nodes）在区间端点附近更密集。这样做的效果不是“消灭误差”，而是让节点乘积和 Lebesgue 函数的最大增长更温和，从而减弱全局多项式在端点附近的振荡。

因此，本章对 Runge 现象的结论是：

* 问题不在于拉格朗日公式本身，而在于节点分布和高次全局插值的组合；
* 等距节点适合低阶或局部插值，但高阶全局插值可能不稳定；
* 切比雪夫节点是全局多项式插值中的重要稳定化选择；
* 如果数据本身有噪声，单纯换成切比雪夫节点也不等于解决问题，此时更应该考虑拟合、正则化或分段方法。


In [ ]:
node_counts = [9, 17, 25]
for n in node_counts:
    equal_nodes = np.linspace(-1.0, 1.0, n)
    cheb_nodes = chebyshev_nodes(-1.0, 1.0, n)

    equal_error = np.max(np.abs(lagrange_interpolate(equal_nodes, runge(equal_nodes), x_plot) - y_true))
    cheb_error = np.max(np.abs(lagrange_interpolate(cheb_nodes, runge(cheb_nodes), x_plot) - y_true))
    print(f"节点数={n:2d}, 等距节点误差={equal_error:10.3e}, 切比雪夫节点误差={cheb_error:10.3e}")

n = 17
x_equal = np.linspace(-1.0, 1.0, n)
x_cheb = chebyshev_nodes(-1.0, 1.0, n)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, y_true, color="black", label="Runge 函数")
plt.plot(x_plot, lagrange_interpolate(x_equal, runge(x_equal), x_plot), label="等距节点")
plt.plot(x_plot, lagrange_interpolate(x_cheb, runge(x_cheb), x_plot), label="切比雪夫节点")
plt.scatter(x_equal, runge(x_equal), s=18, alpha=0.7)
plt.scatter(x_cheb, runge(x_cheb), s=18, alpha=0.7)
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("切比雪夫节点可以减弱端点振荡")
plt.legend()
plt.tight_layout()
plt.show()


## 小结

* 拉格朗日形式直观，适合理解插值基函数。
* 牛顿形式把插值多项式写成递推结构，适合增量添加节点和构造差商表。
* Hermite 插值把导数也纳入插值条件，是 PCHIP、分段三次 Hermite 和样条方法的重要过渡。
* Runge 现象说明，高次全局多项式插值在等距节点上可能严重失效。
* 失效机制可以从节点乘积、Lebesgue 函数和端点附近的误差放大来理解。
* 对有界区间上的光滑函数，切比雪夫节点是重要的稳定化工具。
